In [8]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [9]:
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='auto')
y = y.astype(int)

Binarizamos la matriz y dividimos en datos de entrenamiento y test.

In [10]:
X_binarized = np.where (X>127.5, 1,0)
X_train, X_test, y_train, y_test = train_test_split (X_binarized, y, test_size=0.2, random_state =42 )

Vamos a emplear el modelo de Naive Bayes binarizado para hallar la p(x|y=c)

Creemos primero una función para extraer las imágenes de una clase

In [11]:
def extract_class_images (X, y, class_label):
  mask = y == class_label
  extracted_images = X [mask]
  assert (y[mask]==class_label).all(), "No todas las imágenes extraídas corresponden a la clase solicitada."
  return extracted_images

Función para estimar las probabilidades a priori de cada clase

In [12]:
def calculate_class_priors (y):
  # Contar la aparición de cada clase en el conjunto de etiquetas (y)
  classes, counts = np.unique (y, return_counts=True)
  priors = counts / len(y)
  assert np.isclose (priors.sum(), 1.0), "Las probabilidades a priori no suman 1"
  return priors

Función para estimar los parámetros

In [ ]:
# Utilizaremos el suavizado de Laplace
def get_theta_estimates (X_c):
  N_ic = X_c.sum (axis=0)
  N_c = X_c.shape[0]
  theta_ic_estimates = (N_ic+1)/(N_c+2)
  assert np.all ((theta_ic_estimates >=0)&(theta_ic_estimates <= 1)), "Las probabilidades estimadas deben estar entre 0 y 1"
  return theta_ic_estimates

In [ ]:
def fit_naive_bayes (X,y):
  n_samples, p = X.shape
  classes, _ = np.unique (y, return_counts = True)
  n_classes = len (classes)

  priors_estimates = calculate_class_priors (y)

  thetas_estimates = np.zeros ((p, n_classes))

  for i, c in enumerate (classes):
    X_c = extract_class_images (X=X, y=y, class_label=c)

    class_thetas = get_theta_estimates (X_c)
    thetas_estimates [:,i] = class_thetas

  return priors_estimates, thetas_estimates

In [15]:
priors_estimates, thetas_estimates = fit_naive_bayes (X_train, y_train)

Predecimos las clases. Para calcular $\hat{y}$ utilizamos logaritmos por eficiencia computacional:



\begin{align*}
        & \hat{y} =
        \arg \max_c \left(
        \hat{\pi}_c\prod_{i=1}^{784} \hat{\theta}_{ic}^{x_i}(1- \hat{\theta}_{ic})^{1-x_i}
        \right)
\end{align*}

Para calcular $\hat{y}$ utilizamos logaritmos por eficiencia computacional y claridad de código:




\begin{align*}
  \log \hat{y} &= \log \left(\arg \max_c \left(
  \hat{\pi}_c\prod_{i=1}^{784} \hat{\theta}_{ic}^{x_i}(1- \hat{\theta}_{ic})^{1-x_i}
  \right)\right)\\
  &= \arg \max_c \log \left(
  \hat{\pi}_c\prod_{i=1}^{784} \hat{\theta}_{ic}^{x_i}(1- \hat{\theta}_{ic})^{1-x_i}
  \right)
\end{align*}

Usando las propiedades de los logaritmos:

\begin{align*}
  \log \hat{y} &= \arg \max_c \left(
  \log \hat{\pi}_c + \sum_{i=1}^{784} \left(x_i \log \hat{\theta}_{ic} + (1-x_i) \log (1- \hat{\theta}_{ic})\right)
  \right)
\end{align*}

In [18]:
def predict_naive_bayes (X, priors_estimates, thetas_estimates):
  n_samples = X.shape [0]
  n_classes = len (priors_estimates)

  log_probs = np.zeros ((n_samples, n_classes))

  for i in range (n_samples):
    x = X[i]

    for c in range (n_classes):
      log_prior = np.log (priors_estimates [c])
      log_likelihood = x*np.log (thetas_estimates [:,c])+(1-x)*(1-thetas_estimates[:,c])
      log_likelihood_sum = np.sum (log_likelihood)

      log_prob = log_prior + log_likelihood_sum
      log_probs [i,c] = log_prob

  predicted_classes = np.argmax (log_probs, axis=1)

  return predicted_classes

In [19]:
# Entrenamos nuestro modelo.
priors_estimates, thetas_estimates = fit_naive_bayes (X_train, y_train)

y_pred = predict_naive_bayes (X_test, priors_estimates, thetas_estimates)

print (y_pred)

accuracy = accuracy_score (y_test, y_pred)
print (f"Accuracy:{accuracy}")

[8 9 5 ... 8 7 1]
Accuracy:0.8232857142857143
